---
## 0. Setup

In [0]:
# Environment variables
catalog="main"
schema = "school"
volume="raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
enrollments_path = f"{base_path}/enrollments"
courses_path = f"{base_path}/courses"
enrollments_new_path = f"{base_path}/enrollments-new"

print("Base path:", base_path)
print("Enrollments path:", enrollments_path)
print("Courses path:", courses_path)

Base path: /Volumes/main/school/raw_data
Enrollments path: /Volumes/main/school/raw_data/enrollments
Courses path: /Volumes/main/school/raw_data/courses


In [0]:
# Create structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
print("List structure")

List structure


In [0]:
# Copy data from S3 to Volumes
s3_base = "s3://dalhussein-books/DEA-Book/datasets/school/v1"

dbutils.fs.cp(f"{s3_base}/enrollments", enrollments_path, recurse=True)
dbutils.fs.cp(f"{s3_base}/courses-csv", courses_path, recurse=True)
dbutils.fs.cp(f"{s3_base}/enrollments-new", enrollments_new_path, recurse=True)

print("Data copied successfully")

Data copied successfully


In [0]:
# Create enrollments table (parquet format)
spark.sql(f""" 
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.enrollments 
    AS SELECT * FROM parquet.`{enrollments_path}`
""")

# Create courses table from CSV using PySpark
courses_df = spark.read.csv(courses_path, header=True, inferSchema=True, sep=";")
courses_df.write.mode("ignore").saveAsTable(f"{catalog}.{schema}.courses")

print("Enrollments and courses lists tables")

Enrollments and courses lists tables


In [0]:
# Register the tables as default to use %sql without catalog.schema
spark.sql(f"USE {catalog}.{schema}")
print(f"Using: {catalog}.{schema}")

Using: main.school


---
## 1. The `explode` Function — Transforming Arrays into Rows

The `enrollments` table contains a `courses` column, which is an array of structs.

Let's first explore its structure before applying `explode`.

In [0]:
%sql
-- Preview of enrollments table
-- Note that the 'courses' column is an array of structs
SELECT enroll_id, student_id, courses
FROM enrollments
LIMIT 5;

enroll_id,student_id,courses
000000000003559,S00001,"List(List(C09, 70, 7.2))"
000000000004243,S00002,"List(List(C07, 15, 28.05), List(C06, 90, 2.2))"
000000000004321,S00003,"List(List(C04, 10, 18.0))"
000000000004392,S00004,"List(List(C08, 35, 26.65))"
000000000003673,S00005,"List(List(C01, 75, 12.25), List(C11, 80, 7.6))"


### 1.1 Apply `explode`

The `explode` function transforms each element of the array into an independent row.

The remaining columns (such as `enroll_id` and `student_id`) are **duplicated** for each element.

In [0]:
%sql
-- explode transforms each element of the array into its own row
-- the alias 'course' will name the new column with the resulting struct
SELECT enroll_id, student_id, explode(courses) AS course
FROM enrollments
LIMIT 10

enroll_id,student_id,course
000000000003559,S00001,"List(C09, 70, 7.2)"
000000000004243,S00002,"List(C07, 15, 28.05)"
000000000004243,S00002,"List(C06, 90, 2.2)"
000000000004321,S00003,"List(C04, 10, 18.0)"
000000000004392,S00004,"List(C08, 35, 26.65)"
000000000003673,S00005,"List(C01, 75, 12.25)"
000000000003673,S00005,"List(C11, 80, 7.6)"
000000000004464,S00006,"List(C01, 50, 24.5)"
000000000004464,S00006,"List(C02, 30, 19.6)"
000000000003495,S00007,"List(C09, 60, 9.6)"


### 1.2 Accessing Fields of the Resulting Struct

Once `explode` has been executed, the `course` column is a struct.

We can use the `.` syntax to access its internal fields.

In [0]:
%sql
-- We access the internal fields of the struct using dot notation (.)
SELECT
enroll_id,
student_id,
explode(courses) AS course
FROM enrollments

-- Then in a subquery we can do:
-- course.course_id, course.subtotal, etc.

enroll_id,student_id,course
000000000003559,S00001,"List(C09, 70, 7.2)"
000000000004243,S00002,"List(C07, 15, 28.05)"
000000000004243,S00002,"List(C06, 90, 2.2)"
000000000004321,S00003,"List(C04, 10, 18.0)"
000000000004392,S00004,"List(C08, 35, 26.65)"
000000000003673,S00005,"List(C01, 75, 12.25)"
000000000003673,S00005,"List(C11, 80, 7.6)"
000000000004464,S00006,"List(C01, 50, 24.5)"
000000000004464,S00006,"List(C02, 30, 19.6)"
000000000003495,S00007,"List(C09, 60, 9.6)"


In [0]:
%sql
-- Extract individual fields from the struct using subquery
SELECT 
  enroll_id, 
  student_id, 
  course.course_id, 
  course.subtotal
FROM ( 
  SELECT enroll_id, student_id, explode(courses) AS course 
  FROM enrollments
)
LIMIT 10

enroll_id,student_id,course_id,subtotal
000000000003559,S00001,C09,7.2
000000000004243,S00002,C07,28.05
000000000004243,S00002,C06,2.2
000000000004321,S00003,C04,18.0
000000000004392,S00004,C08,26.65
000000000003673,S00005,C01,12.25
000000000003673,S00005,C11,7.6
000000000004464,S00006,C01,24.5
000000000004464,S00006,C02,19.6
000000000003495,S00007,C09,9.6


---
## 2. Aggregation Functions — `collect_set`, `flatten`, `array_distinct`

Spark SQL offers specialized functions for working with arrays in aggregation contexts.

### 2.1 `collect_set` — Adding Unique Values ​​to an Array

`collect_set` is an aggregation function that returns an array of unique values.

It can even operate on fields within arrays (such as `courses.course_id`).


In [0]:
%sql
-- collect_set groups the unique values ​​into an array per student
-- NOTE: courses_set turns out to be an array of arrays (nested array)
SELECT
    student_id,
    collect_set(enroll_id) AS enrollments_set,
    collect_set(courses.course_id) AS courses_set
FROM enrollments
GROUP BY student_id
LIMIT 10

student_id,enrollments_set,courses_set
S00001,"List(000000000003559, 000000000005067, 000000000005191)","List(List(C09), List(C03, C12), List(C08, C02))"
S00002,"List(000000000004243, 000000000004550, 000000000005192)","List(List(C07, C06), List(C04, C06), List(C02, C06, C01))"
S00003,"List(000000000004321, 000000000004575, 000000000005193)","List(List(C04), List(C04, C10), List(C09, C06))"
S00004,"List(000000000004392, 000000000005022, 000000000005194)","List(List(C08), List(C09, C10), List(C08, C10))"
S00005,"List(000000000003673, 000000000004906, 000000000005195)","List(List(C01, C11), List(C08, C11), List(C09))"
S00006,"List(000000000004464, 000000000004862, 000000000005196)","List(List(C01, C02), List(C09, C06), List(C03, C04))"
S00007,"List(000000000003495, 000000000004660, 000000000005197)","List(List(C09, C02), List(C08, C05), List(C08, C10, C07))"
S00008,"List(000000000004105, 000000000004633, 000000000005198)","List(List(C08, C02), List(C02, C08), List(C05, C07, C12))"
S00009,"List(000000000003825, 000000000005110, 000000000005199)","List(List(C09), List(C03, C11), List(C02, C12))"
S00010,"List(000000000004062, 000000000005040, 000000000005200)","List(List(C09), List(C01, C08, C11), List(C08, C09))"


### 2.2 The Nested Array Problem

When using `collect_set` on `courses.course_id`, we obtain an array of arrays.

This can generate duplicates (the same course_id appears in multiple sub-arrays).

To solve this, we use: `flatten` + `array_distinct`.

In [0]:
%sql
-- `flatten` flattens the nested array -> `array_distinct` removes duplicates
-- We compare the result BEFORE and AFTER applying these functions
SELECT
    student_id,
    collect_set(courses.course_id) AS before_flatten,
    array_distinct(flatten(collect_set(courses.course_id))) AS after_flatten
FROM enrollments
GROUP BY student_id
LIMIT 10

student_id,before_flatten,after_flatten
S00001,"List(List(C09), List(C03, C12), List(C08, C02))","List(C09, C03, C12, C08, C02)"
S00002,"List(List(C07, C06), List(C04, C06), List(C02, C06, C01))","List(C07, C06, C04, C02, C01)"
S00003,"List(List(C04), List(C04, C10), List(C09, C06))","List(C04, C10, C09, C06)"
S00004,"List(List(C08), List(C09, C10), List(C08, C10))","List(C08, C09, C10)"
S00005,"List(List(C01, C11), List(C08, C11), List(C09))","List(C01, C11, C08, C09)"
S00006,"List(List(C01, C02), List(C09, C06), List(C03, C04))","List(C01, C02, C09, C06, C03, C04)"
S00007,"List(List(C09, C02), List(C08, C05), List(C08, C10, C07))","List(C09, C02, C08, C05, C10, C07)"
S00008,"List(List(C08, C02), List(C02, C08), List(C05, C07, C12))","List(C08, C02, C05, C07, C12)"
S00009,"List(List(C09), List(C03, C11), List(C02, C12))","List(C09, C03, C11, C02, C12)"
S00010,"List(List(C09), List(C01, C08, C11), List(C08, C09))","List(C09, C01, C08, C11)"


---
## 3. Join Operations — `INNER JOIN` with Exploded Data

Spark SQL supports all standard join types: `INNER`, `LEFT`, `RIGHT`, `FULL OUTER`, `ANTI`, `CROSS`, `SEMI`.

In this example, we combine the result of an `explode` with the `courses` lookup table
to enrich each record with the course title, instructor, and category.

In [0]:
%sql
-- We first explore the courses table (lookup)
SELECT * FROM courses
LIMIT 5

course_id,title,instructor,category,price
C10,Database Design Solutions,Julia S.,Computer Science,44
C11,Business Intelligence,Tiffany M.,Computer Science,38
C12,Big Data,Bernard M.,Computer Science,30
C01,Data Structures and Algorithms,Tracy N.,Computer Science,49
C02,JavaScript Design Patterns,Ali M.,Computer Science,28


In [0]:
%sql
-- Step 1: Explode to separate each course into its own row
-- Step 2: INNER JOIN with the courses table using course_id as the key
-- We save the result as a VIEW for later reuse
CREATE OR REPLACE VIEW enrollments_enriched AS
SELECT *
FROM (
    SELECT *, explode(courses) AS course
    FROM enrollments
) e
INNER JOIN courses c
ON e.course.course_id = c.course_id

In [0]:
%sql
-- We review the result of the rich join
SELECT 
    enroll_id, 
    student_id, 
    course.course_id, 
    course.subtotal, 
    title, 
    instructor, 
    category
FROM enrollments_enriched
LIMIT 10

enroll_id,student_id,course_id,subtotal,title,instructor,category
000000000003559,S00001,C09,7.2,Advanced Data Structures,Pierre B.,Computer Science
000000000004243,S00002,C07,28.05,Machine Learning,Andriy R.,Computer Science
000000000004243,S00002,C06,2.2,Deep Learning,François R.,Computer Science
000000000004321,S00003,C04,18.0,Robot Dynamics and Control,Mark G.,Computer Science
000000000004392,S00004,C08,26.65,Quantum Computing,Chris N.,Computer Science
000000000003673,S00005,C01,12.25,Data Structures and Algorithms,Tracy N.,Computer Science
000000000003673,S00005,C11,7.6,Business Intelligence,Tiffany M.,Computer Science
000000000004464,S00006,C01,24.5,Data Structures and Algorithms,Tracy N.,Computer Science
000000000004464,S00006,C02,19.6,JavaScript Design Patterns,Ali M.,Computer Science
000000000003495,S00007,C09,9.6,Advanced Data Structures,Pierre B.,Computer Science


---
## 4. Set Operations — `UNION ALL`, `INTERSECT`, `MINUS`

Spark SQL supports set operations for combining, comparing, and isolating datasets.

First, we create a temporary view with **700 new records** to demonstrate the operations.

In [0]:
%sql
-- We create the temporary view with the new enrollment records
CREATE OR REPLACE TEMP VIEW enrollments_updates
AS SELECT * FROM parquet.`/Volumes/main/school/raw_data/enrollments-new`

In [0]:
%sql
-- We check how many records each source has
SELECT 'enrollments' AS source, COUNT(*) AS total FROM enrollments
UNION ALL
SELECT 'enrollments_updates' AS source, COUNT(*) AS total FROM enrollments_updates

source,total
enrollments_updates,700
enrollments,2150


### 4.1 `UNION ALL`: Combine all records (including duplicates)

- `UNION ALL`: Includes **all** records from both sources, preserving duplicates.

- `UNION` (or `UNION DISTINCT`): Returns only **distinct** rows.

In [0]:
%sql
-- UNION ALL: stacks the two datasets including duplicates
-- We expect: enrollments + enrollments_updates = combined total
SELECT COUNT(*) AS total_union_all
FROM ( 
    SELECT * FROM enrollments 
    UNION ALL 
    SELECT * FROM enrollments_updates
)

total_union_all
2850


### 4.2 `INTERSECT` — Common Rows Between Two Datasets

`INTERSECT` returns only the records that **exist in both** sources.

Useful for detecting overlaps or already processed records.

In [0]:
%sql
-- INTERSECT: records that exist in BOTH sources
SELECT COUNT(*) AS total_intersect
FROM ( 
    SELECT * FROM enrollments 
    INTERSECT 
    SELECT * FROM enrollments_updates
)

total_intersect
0


### 4.3 `MINUS`: Records exclusive to the first dataset

`MINUS` (equivalent to `EXCEPT`) returns only the records from the **first dataset** that are **not present** in the second.

Useful for isolating original data before an update.

In [0]:
%sql
-- MINUS: Enrollment records that are NOT in enrollments_updates
-- Gives us the 'original' data before the last insert
SELECT COUNT(*) AS total_minus
FROM (
    SELECT * FROM enrollments
    MINUS
    SELECT * FROM enrollments_updates
)

total_minus
2150


In [0]:
%sql
-- Comparative summary of the three operations
SELECT 'UNION ALL' AS operation, COUNT(*) AS total FROM (SELECT * FROM enrollments UNION ALL SELECT * FROM enrollments_updates)
UNION ALL
SELECT 'INTERSECT' AS operation, COUNT(*) AS total FROM (SELECT * FROM enrollments INTERSECT SELECT * FROM enrollments_updates)
UNION ALL
SELECT 'MINUS' AS operation, COUNT(*) AS total FROM (SELECT * FROM enrollments MINUS SELECT * FROM enrollments_updates)

operation,total
UNION ALL,2850
INTERSECT,0
MINUS,2150


---
## 5. Pivot Tables — The `PIVOT` Clause

The `PIVOT` clause in Spark SQL allows you to **transform values ​​from one column into separate columns**,

applying an aggregation function to each one.

Ideal for creating crosstabs or summarizing data by category.

**General Structure:**
```sql
SELECT * FROM (
    -- subquery with input data
)
PIVOT (
    aggregate_function(column_to_aggregate)
    FOR pivot_column IN ('value1', 'value2', ...)
)
```

In [0]:
%sql
-- We check the available course_ids in the enriched view
SELECT DISTINCT course_id
FROM enrollments_enriched
ORDER BY course_id

course_id
C01
C02
C03
C04
C05
C06
C07
C08
C09
C10


In [0]:
%sql
-- PIVOT: Each course_id becomes a column
-- The cell displays the sum of the subtotal by student and by course
-- NULL means the student is not enrolled in that course
SELECT * FROM (
SELECT
    student_id,
    course.course_id AS course_id,
    course.subtotal AS subtotal
FROM enrollments_enriched
)
PIVOT (
    sum(subtotal) FOR course_id IN (
    'C01', 'C02', 'C03', 'C04', 'C05', 'C06',
    'C07', 'C08', 'C09', 'C10', 'C11', 'C12'
    )
)
LIMIT 10

student_id,C01,C02,C03,C04,C05,C06,C07,C08,C09,C10,C11,C12
S00008,null,37.8,null,null,42.3,null,3.3,34.85,null,null,null,3.0
S00010,19.6,null,null,null,null,null,null,32.8,24.0,null,20.9,null
S00015,24.5,null,null,19.0,null,7.7,null,14.35,1.2,41.8,null,null
S00024,24.5,null,null,null,null,null,null,12.3,27.599999999999998,59.400000000000006,null,null
S00030,null,null,35.0,null,null,null,null,41.0,21.6,28.6,11.4,null
S00031,19.6,15.4,null,null,null,null,3.3,null,30.0,114.4,null,null
S00033,22.05,9.8,null,null,null,null,11.55,null,18.0,17.6,null,10.5
S00042,null,11.2,null,null,null,null,16.5,2.05,6.0,null,19.0,4.5
S00047,null,null,null,null,null,14.3,3.3,14.35,56.400000000000006,null,null,null
S00048,null,null,null,12.0,null,null,null,null,8.4,22.0,22.8,null


### 5.1 PIVOT Breakdown

| Component | Description |
---|---|
**Subquery (lines 1–4)** | Defines the input data: `student_id`, `course_id`, and `subtotal` |
**`sum(subtotal)`** | Aggregation function applied to each pivot value |
**`FOR course_id`** | Column whose values ​​will become new columns |
**`IN ('C01', ...)`** | Explicit list of values ​​to be transformed into columns |
**NULL in cell** | The student is not enrolled in that course |

In [0]:
%sql
-- Variant: PIVOT with COUNT instead of SUM
-- Shows how many times a student is enrolled in each course
SELECT * FROM (
    SELECT
        student_id,
        course.course_id AS course_id
    FROM enrollments_enriched
)
PIVOT (
    count(course_id) FOR course_id IN (
    'C01', 'C02', 'C03', 'C04', 'C05', 'C06',
    'C07', 'C08', 'C09', 'C10', 'C11', 'C12'
    )
)
LIMIT 10

student_id,C01,C02,C03,C04,C05,C06,C07,C08,C09,C10,C11,C12
S00008,null,2,null,null,1,null,1,2,null,null,null,1
S00010,1,null,null,null,null,null,null,2,2,null,1,null
S00015,1,null,null,2,null,1,null,1,1,1,null,null
S00024,1,null,null,null,null,null,null,1,2,2,null,null
S00030,null,null,2,null,null,null,null,1,1,1,1,null
S00031,1,1,null,null,null,null,1,null,2,3,null,null
S00033,1,1,null,null,null,null,1,null,1,1,null,1
S00042,null,1,null,null,null,null,1,1,1,null,2,1
S00047,null,null,null,null,null,1,1,1,3,null,null,null
S00048,null,null,null,1,null,null,null,null,2,1,1,null


In [0]:
def clean_up(): 
    print("Deleting views...") 
    spark.sql("DROP VIEW IF EXISTS enrollments_enriched") 
    spark.sql("DROP VIEW IF EXISTS enrollments_updates") 

    print("Deleting tables...") 
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.enrollments") 
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.courses") 

    print("Deleting files from the volume...") 
    dbutils.fs.rm(enrollments_path, True) 
    dbutils.fs.rm(courses_path, True) 
    dbutils.fs.rm(enrollments_new_path, True) 

    print("Deleting schema...") 
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE") 

    print("Done")

In [0]:
#clean_up()